In [ ]:
import torch
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, kernel_size=3),
            nn.InstanceNorm2d(channels),
            nn.ReLU(inplace=True),

            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, kernel_size=3),
            nn.InstanceNorm2d(channels)
        )

    def forward(self, x):
        return x + self.block(x)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_channels=3):
        super().__init__()
        self.model = nn.Sequential(
            nn.ReflectionPad2d(3),
            nn.Conv2d(input_channels, 64, kernel_size=7),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.InstanceNorm2d(128),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.InstanceNorm2d(256),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
class Decoder(nn.Module):
    def __init__(self, output_channels=3):
        super().__init__()
        self.model = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.InstanceNorm2d(128),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True),

            nn.ReflectionPad2d(3),
            nn.Conv2d(64, output_channels, kernel_size=7),
            nn.Tanh()
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
class Actor(nn.Module):
    def __init__(self, plan_dim=8, input_channels=3, n_res_blocks=4):
        super().__init__()

        # 1. Bring in the Encoder
        self.encoder = Encoder(input_channels)

        # 2. Setup Plan Injection
        self.plan_fc = nn.Linear(plan_dim, 256)

        # 3. Setup the Bottleneck
        self.res_blocks = nn.ModuleList([
            ResidualBlock(256) for _ in range(n_res_blocks)
        ])

        # 4. Bring in the Decoder
        self.decoder = Decoder(input_channels)

    def forward(self, image, plan):
        # 1. Squeeze into features
        features = self.encoder(image)

        # 2. Inject the plan
        projected_plan = self.plan_fc(plan)
        expanded_plan = projected_plan.view(-1, 256, 1, 1)
        x = features + expanded_plan

        # 3. Process
        for block in self.res_blocks:
            x = block(x)

        # 4. Expand to image
        residual_map = self.decoder(x)

        return residual_map

In [ ]:
print("--- Starting Final Test ---")
# 1. Initialize the assembled Actor
my_actor = Actor(plan_dim=8, input_channels=3)

# 2. Create dummy inputs
fake_image = torch.randn(1, 3, 256, 256)
fake_plan = torch.randn(1, 8)

print(f"Feeding Image: {fake_image.shape}")
print(f"Feeding Plan:  {fake_plan.shape}")

# 3. Run the network
final_output = my_actor(fake_image, fake_plan)

# 4. Verify
print("-" * 30)
print(f"Output Shape: {final_output.shape}")

if final_output.shape == (1, 3, 256, 256):
    print("SUCCESS! The modular pipeline works perfectly.")
else:
    print("ERROR in shapes.")

--- Starting Final Test ---
Feeding Image: torch.Size([1, 3, 256, 256])
Feeding Plan:  torch.Size([1, 8])
------------------------------
Output Shape: torch.Size([1, 3, 256, 256])
SUCCESS! The modular pipeline works perfectly.


In [ ]:
import torch
import torchvision.transforms as transforms
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

# ==========================================
# 1. GENERATE A DUMMY IMAGE (No internet needed!)
# ==========================================
print("Generating a test image...")
# Create a 256x256 red canvas
raw_image = Image.new('RGB', (256, 256), color='red')
draw = ImageDraw.Draw(raw_image)
# Draw a blue circle in the middle
draw.ellipse([(64, 64), (192, 192)], fill='blue')

# ==========================================
# 2. PREPARE THE IMAGE
# ==========================================
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])
# Shape becomes (1, 3, 256, 256)
image_tensor = transform(raw_image).unsqueeze(0)

# ==========================================
# 3. RUN THE ACTOR
# ==========================================
print("Running the untrained Actor...")
fake_plan = torch.randn(1, 8)
my_actor = Actor(plan_dim=8, input_channels=3)

with torch.no_grad():
    residual_map = my_actor(image_tensor, fake_plan)

# ==========================================
# 4. THE RL MATH
# ==========================================
final_image_tensor = image_tensor + residual_map
final_image_tensor = torch.clamp(final_image_tensor, 0.0, 1.0)

# ==========================================
# 5. VISUALIZE THE RESULTS
# ==========================================
def tensor_to_image(tensor):
    return tensor.squeeze(0).permute(1, 2, 0).numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(tensor_to_image(image_tensor))
axes[0].set_title("1. Original Image")
axes[0].axis("off")

vis_residual = (residual_map + 1.0) / 2.0
axes[1].imshow(tensor_to_image(vis_residual))
axes[1].set_title("2. Actor Output (Residual)")
axes[1].axis("off")

axes[2].imshow(tensor_to_image(final_image_tensor))
axes[2].set_title("3. Final Result (Distorted)")
axes[2].axis("off")

plt.tight_layout()
plt.show()

Generating a test image...
Running the untrained Actor...


NameError: name 'Actor' is not defined